Week 15 · Day 3 — Vector Databases (FAISS & Chroma)
Why this matters

When docs grow from dozens → millions, in-memory cosine similarity becomes too slow. Vector databases like FAISS (Facebook AI Similarity Search) or Chroma let you index embeddings for fast, scalable retrieval. They’re the backbone of production RAG systems.

Theory Essentials

Problem: O(N) similarity search is too slow at scale.

Vector DB: specialized index structures (e.g., IVF, HNSW) for sub-linear search.

FAISS: C++/Python library for efficient similarity search.

Chroma: user-friendly Python DB for embeddings, good for prototyping.

Workflow:

Embed docs.

Insert into FAISS/Chroma index.

Query → retrieve top-k vectors fast.

In [2]:
# Setup
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# Docs
docs = [
    "The Eiffel Tower is in Paris.",
    "The Colosseum is in Rome.",
    "The Prado Museum is in Madrid.",
    "The Brandenburg Gate is in Berlin.",
    "Big Ben is in London."
]

# Embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(docs).astype("float32")

# 1. Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)   # L2 distance index
index.add(embeddings)

# 2. Query
query = "Where is the Colosseum located?"
q_emb = model.encode([query]).astype("float32")

# 3. Search top-3
D, I = index.search(q_emb, 3)
print("Query:", query)
print("\nTop Retrieved Docs:")
for idx, dist in zip(I[0], D[0]):
    print(f"{docs[idx]} (distance={dist:.3f})")


Query: Where is the Colosseum located?

Top Retrieved Docs:
The Colosseum is in Rome. (distance=0.317)
The Prado Museum is in Madrid. (distance=1.451)
The Brandenburg Gate is in Berlin. (distance=1.559)


1) Core (10–15 min)

Task: Change the query to “Where is Big Ben?” and retrieve top-2 results.

In [4]:
query = "Where is Big Ben?"
q_emb = model.encode([query]).astype("float32")
D, I = index.search(q_emb, 2)
for idx in I[0]:
    print(docs[idx])


Big Ben is in London.
The Prado Museum is in Madrid.


2) Practice (10–15 min)

Task: Print distances alongside results and see if lower distance = better match.

In [5]:
for idx, dist in zip(I[0], D[0]):
    print(docs[idx], "→ distance:", dist)


Big Ben is in London. → distance: 0.30784696
The Prado Museum is in Madrid. → distance: 1.7117897


3) Stretch (optional, 10–15 min)

Task: Add a new doc (“The Acropolis is in Athens”), re-embed, and re-add to FAISS. Query “Acropolis.”

In [6]:
docs.append("The Acropolis is in Athens.")
new_emb = model.encode([docs[-1]]).astype("float32")
index.add(new_emb)

query = "Where is the Acropolis?"
q_emb = model.encode([query]).astype("float32")
D, I = index.search(q_emb, 1)
print(docs[I[0][0]])


The Acropolis is in Athens.


Mini-Challenge (≤40 min)

Build a Scalable Retrieval Index

Create an index with at least 10+ documents.

Write search_faiss(query, k=3) that returns top-k docs + distances.

Test with at least 3 queries.

Acceptance Criteria:

Results returned in under a second.

Distances show the correct doc has the lowest score.

Function works for new docs added dynamically.

In [11]:
# Setup
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# Docs
docs = [
    "Ilia Topuria is the pound-for-pound number one.",
    "Ilia Topuria is the UFC lightweight champion.",
    "Alexander Volkanovski is the UFC featherweight champion.",
    "Merab Dvalishvili is the UFC bantamweight champion.",
    "Khamzat Chimaev is the UFC middleweight champion.",
    "Tom Aspinall is the UFC heavyweight champion.",
    "Conor McGregor is a former two-division UFC champion at featherweight and lightweight.",
    "Alexandre Pantoja is the UFC flyweight champion.",
    "Jack Della Maddalena is the UFC welterweight champion.",
    "Magomed Ankalaev is the UFC light heavyweight champion."
]

# Embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(docs).astype("float32")

# 1. Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)   # L2 distance index
index.add(embeddings)

# 2. Query

for query in ["Who is Ilia?", "Who is the current UFC middleweight champion?", "Who was Connor?"]:

    q_emb = model.encode([query]).astype("float32")

    # 3. Search top-3
    D, I = index.search(q_emb, 3)
    print("Query:", query)
    print("\nTop Retrieved Docs:")
    for idx, dist in zip(I[0], D[0]):
        print(f"{docs[idx]} (distance={dist:.3f})")
    print()

Query: Who is Ilia?

Top Retrieved Docs:
Ilia Topuria is the pound-for-pound number one. (distance=0.841)
Ilia Topuria is the UFC lightweight champion. (distance=0.884)
Alexandre Pantoja is the UFC flyweight champion. (distance=1.575)

Query: Who is the current UFC middleweight champion?

Top Retrieved Docs:
Khamzat Chimaev is the UFC middleweight champion. (distance=0.600)
Conor McGregor is a former two-division UFC champion at featherweight and lightweight. (distance=0.703)
Alexander Volkanovski is the UFC featherweight champion. (distance=0.736)

Query: Who was Connor?

Top Retrieved Docs:
Conor McGregor is a former two-division UFC champion at featherweight and lightweight. (distance=1.550)
Alexander Volkanovski is the UFC featherweight champion. (distance=1.663)
Tom Aspinall is the UFC heavyweight champion. (distance=1.671)



Notes / Key Takeaways

In-memory search = fine for toy examples, not for scale.

FAISS supports many index types (FlatL2, IVFFlat, HNSW).

Chroma = higher-level DB API (can store metadata + persist).

Core idea: embed once → search many times, fast.

Vector DBs = backbone of modern RAG apps (chat with PDFs, enterprise docs).

Reflection

Why is L2 distance (or cosine similarity) used instead of keyword matching?

What trade-off do approximate search methods (HNSW, IVF) make?

Why is L2 distance (or cosine similarity) used instead of keyword matching?

Keyword matching only looks at exact words → it misses synonyms, phrasing differences, or semantic meaning.

L2 / cosine compares embeddings, which capture meaning in vector space, so “capital of France” and “Paris city” are considered close even without shared words.

What trade-off do approximate search methods (HNSW, IVF) make?

Exact search = perfect accuracy but slower, especially with millions of vectors.

Approximate methods (HNSW, IVF) trade a little accuracy (you might not always get the absolute nearest neighbor) for much higher speed and scalability.